# Capstone Project: Academic Research Assistant
This notebook demonstrates a generative AI-powered research assistant that helps users explore academic papers more efficiently. Users can ask open-ended questions, receive grounded answers with citations, and evaluate the quality of those answers using automated scoring.

The assistant is built using Retrieval-Augmented Generation (RAG), where relevant information is retrieved from a large dataset of academic abstracts and passed to a Gemini model for response generation. The assistant supports multi-turn conversations, response tone customization, and optional evaluation breakdowns.

# Gen AI Capabilities Used
This project showcases multiple Gen AI capabilities from the list provided in the course:

* **Retrieval-Augmented Generation (RAG)** – uses Gemini and vector search to answer user questions grounded in source data.

* **Embeddings** – academic abstracts and queries are embedded using Gemini’s embedding model.

* **Structured Output (JSON Mode)** – Gemini is prompted to return evaluation scores as JSON for programmatic parsing.

* **Gen AI Evaluation** – answers are automatically rated on relevance, grounding, and clarity, with detailed explanations.

* **Grounding** – responses cite specific document chunks using a segment reference style.

## Setup: Dependencies and Imports
This section installs and imports all required Python libraries used throughout the notebook.

* google-generativeai is used to access Gemini for embeddings, generation, and evaluation.

* chromadb is removed if present to avoid dependency conflicts.

* Standard packages like numpy, pandas, and os are used for data manipulation and file handling.

* sklearn.metrics.pairwise.cosine_similarity is used to compute vector similarity between queries and document chunks.

If running in a Kaggle environment, the Gemini API key is loaded securely using the UserSecretsClient.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


In [2]:
!pip uninstall -qqy jupyterlab kfp  # Remove unused conflicting packages
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"
!pip install -q google-generativeai
!pip install kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.3 MB/s eta 0:00

In [3]:
from google import genai
from google.genai import types

from IPython.display import Markdown
from kaggle_secrets import UserSecretsClient

genai.__version__

'1.7.0'

### Set up your API key

To run the following cell, your API key must be stored it in a [Kaggle secret](https://www.kaggle.com/discussions/product-feedback/114053) named `GOOGLE_API_KEY`.

If you don't already have an API key, you can grab one from [AI Studio](https://aistudio.google.com/app/apikey). You can find [detailed instructions in the docs](https://ai.google.dev/gemini-api/docs/api-key).

To make the key available through Kaggle secrets, choose `Secrets` from the `Add-ons` menu and follow the instructions to add your key or enable it for this notebook.

Next, you will need to add your API key to your Kaggle Notebook as a Kaggle User Secret.

![](https://storage.googleapis.com/kaggle-media/Images/5dgai_1.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_2.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_3.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_4.png)

In [4]:
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

In [5]:
client = genai.Client(api_key=GOOGLE_API_KEY)

for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
m

In [6]:
# Set Kaggle API key - assumes you've uploaded your kaggle.json to the Kaggle Notebook
import google.generativeai as genai
os.environ['KAGGLE_CONFIG_DIR'] = '/kaggle/input/'

In [7]:
from IPython.display import Markdown
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import math
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

 ## Data Preparation: Load and Chunk Academic Abstracts
This section loads a large dataset of academic papers from arXiv and prepares it for use with the research assistant.

* The notebook loads a subset of the full dataset (arxiv-metadata-oai-snapshot.json) to stay within resource limits.

* Missing abstracts and those with fewer than 100 characters are filtered out.

* Each abstract is split into overlapping text chunks to support fine-grained retrieval and grounded responses.

* Metadata such as paper title, categories, update date, and chunk index is preserved for downstream citation formatting.

Chunking ensures that relevant slices of a long abstract can be matched and cited in response to specific user queries.

In [8]:
# >>> Load a manageable sample from the large arXiv dataset
# The dataset is in JSONL format (1 paper per line). We stream it in chunks to avoid memory issues.
sample_rows = 2000
df = pd.read_json("/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json", lines=True, chunksize=sample_rows)

# >>> Extract the first chunk (batch of 2000 entries)
df = next(df)

# >>> Keep only the essential fields for academic question answering
# We also filter out rows with missing values or abstracts that are too short to be meaningful.
df = df[["id", "title", "abstract", "categories", "update_date"]].dropna()
df = df[df["abstract"].str.len() > 100]

# >>> Preview the processed dataset
df.head()


,id,title,abstract,categories,update_date
0,704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,hep-ph,2008-11-26
1,704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",math.CO cs.CG,2008-12-13
2,704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,physics.gen-ph,2008-01-13
3,704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,math.CO,2007-05-23
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,math.CA math.FA,2013-10-15


In [9]:
# >>> Define a utility function to split long text into overlapping chunks
def chunk_text(text, chunk_size=1000, overlap=200):
    """
    Splits input text into overlapping chunks of specified length.

    This is important for retrieval-based models like RAG, which work best
    when context is semantically coherent and under a model’s max token limit.
    
    Parameters:
    - text (str): The full string to be chunked.
    - chunk_size (int): Maximum number of characters per chunk.
    - overlap (int): Number of characters that each chunk overlaps with the previous.
    
    Returns:
    - List[str]: List of text chunks.
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # Move forward with overlap
    return chunks


# >>> Apply chunking to each abstract and collect metadata for downstream use
chunked_data = []

# Iterate over each paper in the original dataset
for _, row in df.iterrows():
    abstract_chunks = chunk_text(row["abstract"])
    
    # Store each chunk with its metadata for use in embedding + retrieval
    for i, chunk in enumerate(abstract_chunks):
        chunked_data.append({
            "id": row["id"],
            "title": row["title"],
            "chunk": chunk,
            "chunk_index": i,
            "categories": row["categories"],
            "update_date": row["update_date"]
        })

# >>> Convert chunked results into a structured DataFrame
chunk_df = pd.DataFrame(chunked_data)

# Preview the structure: one row per text chunk with associated metadata
chunk_df.head()


,id,title,chunk,chunk_index,categories,update_date
0,704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,0,hep-ph,2008-11-26
1,704.0001,Calculation of prompt diphoton production cros...,of a Higgs\nboson are contrasted with those pr...,1,hep-ph,2008-11-26
2,704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",0,math.CO cs.CG,2008-12-13
3,704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,0,physics.gen-ph,2008-01-13
4,704.0003,The evolution of the Earth-Moon system based o...,o slowing with the angular acceleration rate\n...,1,physics.gen-ph,2008-01-13


## Embedding Abstract Chunks with Gemini
To enable similarity-based retrieval, each chunk of academic text is converted into a vector embedding using Google’s Gemini embedding model.

* The get_embedding() function sends each chunk to Gemini’s embedding-001 model.

* The returned vector (768-dimensional) numerically represents the meaning of the chunk.

* Embeddings are stored in a new column and used later to compare against user questions.

This allows the assistant to retrieve semantically similar chunks, even if wording differs from the user's query.

In [10]:
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm
tqdm.pandas()

# >>> Securely load and configure your Gemini API key for use
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

# >>> Define a helper function to get an embedding for a given text chunk
def get_embedding(text):
    """
    Sends a string of text to the Gemini embedding model and returns a 768-dimensional vector.

    Parameters:
    - text (str): A single chunk of academic text (e.g., part of an abstract).

    Returns:
    - List[float]: A list of 768 floats representing the semantic embedding.
    """
    try:
        response = genai.embed_content(
            model="models/embedding-001",
            content=text,
            task_type="retrieval_document"
        )
        return response["embedding"]
    except Exception as e:
        print(f"[Embedding error] {e}")
        return None

# >>> Generate and store embeddings for each chunk
# This powers vector search for relevant academic content
chunk_df["embedding"] = chunk_df["chunk"].progress_apply(get_embedding)

# >>> Filter out any chunks where embedding failed (e.g., API issue or empty input)
chunk_df = chunk_df[chunk_df["embedding"].notnull()]


100%|██████████| 2933/2933 [14:30<00:00,  3.37it/s]


## Similarity Search: Retrieve Relevant Chunks
This section defines the retrieval function used to find the most relevant document chunks for a given user query.

* The retrieve_similar_chunks() function embeds the user’s question using Gemini’s embedding model.

* Cosine similarity is calculated between the query embedding and all precomputed abstract chunk embeddings.

* The top-k most similar chunks are selected and returned with their associated metadata.

This retrieval step powers the “retrieval” in Retrieval-Augmented Generation (RAG), ensuring answers are grounded in relevant academic content.

In [11]:
# >>> Convert list of individual embeddings into a matrix for efficient vector search
embedding_matrix = np.array(chunk_df["embedding"].tolist())

# >>> Define a function to find the most relevant chunks for a given query
def retrieve_similar_chunks(query, top_k=5):
    """
    Embeds a user query using Gemini and retrieves the top-k most similar document chunks
    based on cosine similarity.

    This is the core of the retrieval phase in a RAG pipeline.
    
    Parameters:
    - query (str): The user's natural-language research question.
    - top_k (int): Number of most relevant chunks to return.

    Returns:
    - DataFrame: Top-k most similar chunks, each with metadata and similarity score.
    """
    # Step 1: Embed the query using the Gemini embedding model
    try:
        query_embedding = genai.embed_content(
            model="models/embedding-001",
            content=query,
            task_type="retrieval_query"
        )["embedding"]
    except Exception as e:
        print(f"[Query Embedding Error] {e}")
        return pd.DataFrame()

    # Step 2: Compute cosine similarity between the query and all abstract chunks
    query_vector = np.array(query_embedding).reshape(1, -1)
    similarities = cosine_similarity(query_vector, embedding_matrix)[0]

    # Step 3: Rank chunks by similarity and return the top-k most relevant
    chunk_df["similarity"] = similarities
    top_matches = chunk_df.sort_values(by="similarity", ascending=False).head(top_k)

    return top_matches.reset_index(drop=True)

# >>> Example query: retrieving relevant papers from the vector store
query = "What are the latest methods used in reinforcement learning?"
top_chunks = retrieve_similar_chunks(query)

# Display titles and snippets of the most relevant chunks
top_chunks[["title", "chunk", "similarity"]]


,title,chunk,similarity
0,Real Options for Project Schedules (ROPS),Real Options for Project Schedules (ROPS) ha...,0.613026
1,Behavioral response to strong aversive stimuli...,havior has been formulated in terms of\nexisti...,0.603006
2,"""Illusion of control"" in Minority and Parrondo...","er words, low-entropy (more informative)\nstra...",0.601188
3,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...,0.598512
4,Probability distributions generated by fractio...,Fractional calculus allows one to generalize...,0.596716


## Prompt Construction with Tone Control and Citations
This section defines the function that builds the prompt sent to Gemini for answer generation.

* The build_rag_prompt() function combines:

    * The user’s research question,

    * Retrieved context chunks, and

    * Metadata-formatted citations.

* Users can choose from three tone presets: "academic", "friendly", or "critical".

* Inline citations follow a consistent structure using document metadata and segment numbers.

This allows for structured, grounded, and stylistically flexible responses — ideal for adapting the assistant to different research audiences.

In [12]:
def build_rag_prompt(query, retrieved_chunks, tone="academic"):
    """
    Builds a Gemini-ready prompt for answering a user question using tone presets,
    structured context injection, and academic citation formatting.

    Parameters:
    - query (str): The user’s natural-language research question.
    - retrieved_chunks (DataFrame): Top-k relevant chunks with metadata.
    - tone (str): Response tone setting ("academic", "friendly", or "critical").

    Returns:
    - prompt (str): A fully formatted prompt to send to Gemini.
    - citation_list (List[str]): A list of references for footer-style citation display.
    """
    context_blocks = []
    citation_list = []

    # >>> Format each retrieved chunk with its source metadata for grounded generation
    for i, row in retrieved_chunks.iterrows():
        citation = f'“{row["title"]}” ({row["update_date"]}, {row["categories"]}) [Segment #{row["chunk_index"]}]'
        block = f'Source: {citation}\n{row["chunk"]}'
        context_blocks.append(block)
        citation_list.append(citation)

    # Combine all sources into one RAG-style context block
    context = "\n\n".join(context_blocks)

    # 🎭 Tone-controlled system instruction
    if tone == "friendly":
        instruction = "Respond in a helpful, casual tone like a study buddy. Use friendly phrases and emojis if you like."
    elif tone == "critical":
        instruction = "Respond like a critical peer reviewer. Be skeptical, precise, and highlight any limitations or weaknesses."
    else:  # academic (default)
        instruction = "Respond in a formal academic tone. Use citations and structured, objective analysis."

    # Final structured prompt to send to Gemini
    prompt = f"""{instruction}

Excerpts:
{context}

User Question:
{query}

Please include:
- A clear, thoughtful response
- In-text citations like (Source: “...” [Segment #X])
- A follow-up suggestion if helpful
"""
    return prompt, citation_list


## Response Generation with Gemini
This section defines the function responsible for generating answers based on the user’s query and retrieved context.

* The generate_rag_response() function sends the full prompt to the gemini-1.5-flash model.

* Gemini responds with a complete, citation-aware answer tailored to the chosen tone.

* This step finalizes the RAG loop: after retrieval and prompt construction, generation produces the assistant’s reply.

This function ensures that responses are not only relevant, but also well-structured and backed by retrieved source material.

In [13]:
# >>> Generate a grounded answer using Gemini and retrieved academic context
def generate_rag_response(prompt):
    """
    Uses Gemini (via `models/gemini-1.5-flash`) to generate a response to a user question,
    grounded in the provided academic context (RAG-style).

    Parameters:
    - prompt (str): A structured prompt containing both the context and the user's question.

    Returns:
    - str: Gemini-generated answer as a string.
    """
    try:
        model = genai.GenerativeModel("models/gemini-1.5-flash")
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"[Generation Error] {e}")
        return "Error generating response."


## Suggested Questions from Sample Abstracts
This section defines a utility that helps users get started by generating example research questions.

* The suggest_questions() function samples a few abstracts and uses Gemini to propose academic-style questions based on their content.

* This is useful for users who are unfamiliar with the dataset or unsure what to ask.

* Suggested questions are displayed when the user types "help" or submits a blank prompt.

This feature improves accessibility and supports user discovery by showing the assistant’s capabilities in action.

In [14]:
def suggest_questions(sample_df, n=3):
    """
    Generates example academic research questions using Gemini,
    based on randomly selected titles and abstracts from the dataset.

    This helps users unfamiliar with the content get started by
    surfacing relevant and context-aware example queries.

    Parameters:
    - sample_df (DataFrame): The dataset of papers (e.g., full or chunked abstract DataFrame).
    - n (int): Number of sample questions to generate.

    Returns:
    - List[str]: A list of suggested user questions.
    """
    sample = sample_df.sample(n)
    questions = []

    for _, row in sample.iterrows():
        title = row["title"]
        abstract = row["abstract"]

        # Construct a prompt to convert title + abstract into a relevant research question
        suggestion_prompt = (
            f"Based on the following paper title and abstract, suggest one academic research question:\n\n"
            f"Title: {title}\nAbstract: {abstract}\n\nQuestion:"
        )

        # Use Gemini to generate a sample question
        model = genai.GenerativeModel("models/gemini-1.5-flash")
        response = model.generate_content(suggestion_prompt)
        questions.append(response.text.strip())

    return questions



## Evaluating Answer Quality with Gemini
This section defines a two-step process to evaluate the quality of the assistant’s responses.

1. Score Generation
Gemini is prompted to return structured scores (1–5) for:

    * Relevance: How well the answer matches the user’s question

    * Grounding: Whether the answer is based on provided sources

    * Clarity: How clearly the answer is written
    
    The scores are returned in JSON format for programmatic parsing.

2. Explanation of Scores
Gemini is then prompted to explain the reasoning behind each score in plain language.

This evaluation pipeline helps users understand not only how well an answer performed, but also why — demonstrating GenAI reasoning and structured output handling.

In [15]:
import json
import re
import google.generativeai as genai

# >>> Helper function: extract valid JSON block from Gemini's response
def extract_json_block(text):
    """
    Extracts and parses the first JSON object found in a string,
    even if it's wrapped in Markdown-style formatting (e.g., ```json ... ```).

    Parameters:
    - text (str): The raw text output from Gemini.

    Returns:
    - dict or None: Parsed JSON object, or None if extraction fails.
    """
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError as e:
            print(f"[JSON Decode Error] {e}")
    return None


# >>> Gemini-based scoring + explanation generator
def evaluate_and_explain_answer(answer, query, context):
    """
    Step 1: Ask Gemini to assign numeric scores (1–5) for relevance, grounding, and clarity.
    Step 2: Ask Gemini to explain those scores in natural language.

    This function supports both GenAI evaluation and structured output goals for the capstone.

    Parameters:
    - answer (str): The assistant’s response to be evaluated.
    - query (str): The user’s original question.
    - context (str): The academic chunks used to generate the answer.

    Returns:
    - score_json (dict): Dictionary of {relevance, grounding, clarity}.
    - explanation_text (str): Natural language breakdown of the scoring.
    """
    model = genai.GenerativeModel("models/gemini-1.5-flash")

    # Step 1: Ask for structured scoring output (in strict JSON format)
    scoring_prompt = f"""
You are evaluating an assistant's academic answer to a user research question.
Return a JSON object with integer scores from 1 to 5 for the following:
- Relevance
- Grounding
- Clarity

Do NOT include any explanation text. Return only this format:
{{
  "relevance": <int>,
  "grounding": <int>,
  "clarity": <int>
}}

--- Assistant Response ---
{answer}

--- User Question ---
{query}

--- Source Context ---
{context}
"""

    try:
        score_response = model.generate_content(scoring_prompt)
        raw_score_text = score_response.text.strip()

        score_json = extract_json_block(raw_score_text)
        if not score_json:
            raise ValueError("No valid JSON object found in Gemini's response.")

    except Exception as e:
        print(f"[Scoring Error] {e}")
        print("⚠️ Gemini likely returned invalid or empty text.")
        return None, None

    # Step 2: Ask Gemini to explain the assigned scores in an academic tone
    explanation_prompt = f"""
Let's evaluate the assistant's response using the scores below.

Relevance Score: {score_json['relevance']}
Grounding Score: {score_json['grounding']}
Clarity Score: {score_json['clarity']}

Explain in clear, academic language why each score was given. Separate each section clearly.

--- Assistant Response ---
{answer}

--- User Question ---
{query}

--- Source Context ---
{context}
"""

    try:
        explanation_response = model.generate_content(explanation_prompt)
        explanation_text = explanation_response.text.strip()
    except Exception as e:
        print(f"[Explanation Error] {e}")
        explanation_text = "Explanation generation failed."

    return score_json, explanation_text


In [16]:
import re

def extract_score(category, text):
    """
    Legacy utility to extract a numeric score (1–5) from Gemini's
    natural language evaluation output.

    This regex parser searches for a score associated with a given category
    (e.g., "Relevance", "Grounding", or "Clarity") in markdown-style output.

    Parameters:
    - category (str): The scoring category (e.g., "Relevance").
    - text (str): The Gemini-generated explanation block.

    Returns:
    - int or None: Parsed score, or None if no match is found.
    """
    pattern = rf"\*\*?{category}.*?Score:?\s*(\d)/5"
    match = re.search(pattern, text, re.IGNORECASE)
    return int(match.group(1)) if match else None


## ResearchAgent Class: Multi-Turn Academic Assistant
This section defines the main ResearchAgent class, which manages all interactions with the assistant.

The agent supports:

* Multi-turn memory – stores recent user/assistant exchanges for optional summarization.

* Retrieval-Augmented Generation (RAG) – fetches and grounds answers in relevant source content.

* Response tone control – academic, friendly, or critical tone depending on user preference.

* On-demand evaluation – allows users to request quality scoring and detailed explanations.

* Session summary and scoring – summarizes the conversation and displays average evaluation metrics at the end.

This modular class wraps all assistant logic, making it easy to extend or embed into future applications.

In [17]:
# >>> Academic Research Agent: Handles multi-turn questions using RAG + Gemini
class ResearchAgent:
    """
    A lightweight assistant that handles academic queries using retrieval-augmented generation (RAG),
    powered by Gemini. Supports multi-turn memory, evaluation, tone presets, and summarization.
    """

    def __init__(self, retriever_func, generator_func, max_turns=5, tone="academic"):
        """
        Initialize the assistant with a retriever and response generator.

        Parameters:
        - retriever_func (function): Retrieves relevant document chunks.
        - generator_func (function): Generates a response using a prompt.
        - max_turns (int): Maximum number of memory turns to retain.
        - tone (str): Default tone to use for all responses (academic, friendly, or critical).
        """
        self.retriever = retriever_func
        self.generator = generator_func
        self.memory = []
        self.max_turns = max_turns
        self.eval_scores = []
        self.tone = tone

    def ask(self, user_query):
        """
        Handles a single user question:
        - Retrieves relevant context using embeddings
        - Builds and sends prompt to Gemini
        - Stores the response in memory
        - Optionally evaluates and explains the output

        Parameters:
        - user_query (str): The research question from the user.

        Returns:
        - str: Assistant's response (also printed to console).
        """
        # >>> Retrieve top-matching content using vector search
        top_chunks = self.retriever(user_query)
        prompt, citation_list = build_rag_prompt(user_query, top_chunks, self.tone)
        answer = self.generator(prompt)

        # >>> Save this interaction to memory
        self.memory.append({"user": user_query, "assistant": answer})
        if len(self.memory) > self.max_turns:
            self.memory.pop(0)

        # >>> Display the generated answer
        print(f"\n🤖 Assistant:\n{answer}\n")

        # >>> Display footer-style citations
        print("📚 Sources Referenced:")
        for i, citation in enumerate(set(citation_list), start=1):
            print(f"{i}. {citation}")
        print()

        # >>> Ask user if they’d like to evaluate the response
        choice = input("🧪 Would you like me to evaluate this response? (yes/no): ").strip().lower()
        if choice in ["yes", "y"]:
            context_text = "\n\n".join(top_chunks["chunk"].tolist())
            eval_scores, explanation = evaluate_and_explain_answer(answer, user_query, context_text)

            if eval_scores:
                self.eval_scores.append(eval_scores)

                print("📊 Evaluation Scores:")
                print(f"- Relevance: {eval_scores['relevance']}")
                print(f"- Grounding: {eval_scores['grounding']}")
                print(f"- Clarity:   {eval_scores['clarity']}\n")

                # >>> Optional: Explain the reasoning behind each score
                explain = input("🧠 Would you like a breakdown of these scores? (yes/no): ").strip().lower()
                if explain in ["yes", "y"]:
                    print("\n📝 Evaluation Explanation:\n")
                    print(explanation)

        return answer

    def show_memory(self):
        """
        Displays all previous user-assistant turns stored in memory.
        """
        for i, turn in enumerate(self.memory):
            print(f"Turn {i+1} — User: {turn['user']}")
            print(f"         Assistant: {turn['assistant']}\n")

    def summarize_memory(self):
        """
        Summarizes the full assistant-user conversation using Gemini.
        Designed to be called at the end of a chat session.
        """
        if not self.memory:
            print("🧠 No memory to summarize.")
            return

        # >>> Construct prompt from full memory
        summary_prompt = "Summarize the key questions and answers from the following research conversation:\n\n"
        for turn in self.memory:
            summary_prompt += f"User: {turn['user']}\nAssistant: {turn['assistant']}\n\n"

        try:
            model = genai.GenerativeModel("models/gemini-1.5-flash")
            response = model.generate_content(summary_prompt)
            print("\n🧠 Conversation Summary:\n")
            print(response.text)
        except Exception as e:
            print(f"[Summary Error] {e}")

    def show_average_scores(self):
        """
        Averages all stored evaluation scores and prints the result.
        Useful for reflecting on assistant performance across the session.
        """
        if not self.eval_scores:
            print("📊 No evaluations were recorded.")
            return

        total = {"relevance": 0, "grounding": 0, "clarity": 0}
        for score in self.eval_scores:
            for k in total:
                total[k] += score[k]

        n = len(self.eval_scores)
        avg = {k: round(total[k] / n, 2) for k in total}

        print("\n📈 Average Evaluation Scores Across Session:")
        print(f"- Relevance: {avg['relevance']}")
        print(f"- Grounding: {avg['grounding']}")
        print(f"- Clarity:   {avg['clarity']}")


## Running the Assistant: Interactive Chat Loop
This section launches the assistant and simulates a real-time research conversation.

* Users are prompted to select a response tone (academic, friendly, or critical).

* The assistant then enters a multi-turn loop, allowing users to ask questions, request evaluations, and explore academic content.

* At the end of the session, users are optionally offered:

    * A conversation summary generated by Gemini

    * A display of their average evaluation scores

This loop showcases the full functionality of the assistant — combining retrieval, generation, evaluation, and summarization into a complete user experience.

In [18]:
# >>> Initialize the academic assistant with retrieval and generation functions
agent = ResearchAgent(
    retriever_func=retrieve_similar_chunks,
    generator_func=generate_rag_response
)

# >>> Simulated Chat Interface: Loop-based assistant with user input
def run_chat(agent, max_turns=10):
    """
    Launches a multi-turn conversation with the academic assistant.

    This function:
    - Prompts for tone selection (academic, friendly, or critical)
    - Resets memory between sessions
    - Provides help suggestions if user is unsure what to ask
    - Loops until user opts to exit or max turns is reached
    - Optionally summarizes and evaluates the session at the end
    """

    # 🔄 Reset state to start a fresh session
    agent.memory = []
    agent.eval_scores = []

    # 🎭 Let the user pick the assistant's tone
    tone = input("🎭 Choose response style (academic, friendly, critical): ").strip().lower()
    if tone not in ["academic", "friendly", "critical"]:
        tone = "academic"
    agent.tone = tone
    print(f"✅ Tone set to: {tone.capitalize()}\n")

    # 🧠 Introductory message and prompt guidance
    print("🧠 Academic Research Assistant is ready! Please type a question to continue. Type 'exit' to quit.\n")
    print("💬 You can ask a question anytime. Type 'help' or press enter for suggestions.\n")

    turn = 0
    while turn < max_turns:
        user_query = input("🧑‍🎓 You: ")

        # If the user asks for help or submits a blank query, suggest questions
        if not user_query.strip() or user_query.lower().strip() == "help":
            print("\n🧭 Here are a few sample questions based on your dataset:\n")
            for q in suggest_questions(df):  # Assumes 'df' is the full abstracts DataFrame
                print(f"- {q}")
            continue

        # Exit condition
        if user_query.strip().lower() in ['exit', 'quit']:
            print("\n👋 Ending chat. Thanks for using the research assistant!")
            break

        # Process the user's query via RAG pipeline
        agent.ask(user_query)
        turn += 1

        # Ask if the user wants to continue
        follow_up = input("\n🔁 Would you like to ask another question? (yes/no): ").strip().lower()
        if follow_up not in ["yes", "y"]:
            print("\n👋 Conversation ended. Have a great day!")

            # Optional: Ask user if they want a summary of their session
            show_summary = input("📝 Would you like a summary of your conversation? (yes/no): ").strip().lower()
            if show_summary in ["yes", "y"]:
                agent.summarize_memory()

            # Always show average evaluation scores
            agent.show_average_scores()
            break


## Moment of Truth: Run the Assistant
You’ve made it. After embedding, chunking, retrieving, prompting, evaluating, and formatting like a champion… it all comes down to this:

### run_chat(agent)
    
Your academic research assistant is ready to roll — and it even remembers what you say (temporarily, of course).

Whether you’re hunting for insights, testing its judgment, or just vibing with citations, this is where it all comes together.

Go ahead. Uncomment it and Press Run.

##### **You’ve earned it.**

In [19]:
#run_chat(agent)

## Acknowledgments & References
This project was developed as part of the [Google Generative AI for Developers Capstone Project], hosted on Kaggle.

### Datasets:

* arXiv metadata provided by Cornell University on Kaggle (CC0-1.0 license)

### Models & APIs:

* All embeddings, generations, and evaluations were powered by Google’s Gemini API

    * models/embedding-001 for semantic vector representation

    * models/gemini-1.5-flash for content generation and evaluation

### Python Libraries:

* pandas and numpy for data processing

* sklearn.metrics.pairwise.cosine_similarity for vector similarity

* tqdm for progress tracking

* re and json for parsing structured output

* IPython.display.Markdown for enhanced formatting in output

* os for file system navigation

* kaggle_secrets.UserSecretsClient for secure API key handling

**Special thanks to the course instructors and the open-source community for enabling this work. This assistant was built for educational purposes and demonstrates the real-world potential of generative AI in academic support tools.**